In [ ]:
"""
=============================================================================
ETL PIPELINE: BRAZILIAN RETAIL CREDIT BASE CONSTRUCTION (SCR/BCB)
Senior Quant Review: Out-of-Core Processing, Survivorship Bias Prevention,
Memory Optimization (Downcast/Category), and Explicit Date Parsing.
Author: Itaiguara de Oliveira Bezerra
=============================================================================
"""

import glob
import logging
import os
import unicodedata
import zipfile
from typing import Optional

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# ==============================================================================
# GENERAL SETTINGS AND CONSTANTS
# ==============================================================================
logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

ZIP_FOLDER = "dados_scr/"
PARQUET_OUTPUT = "painel_scr_pf_bruto.parquet"

# Original Portuguese terms expected in the Central Bank CSV files
BALANCE_TERMS = [
    "carteira", "vencido", "inadimplencia", "inadimplente", 
    "ativo", "saldo", "valor", "a_vencer", "prejuizo", "provisao"
]

CATEGORICAL_COLUMNS = ["uf", "cnae_ocupacao", "modalidade", "instituicao", "porte", "indexador"]

# ==============================================================================
# DATA ENGINEERING FUNCTIONS
# ==============================================================================
def normalize_column_name(col: str) -> str:
    """Removes accents, special characters, and standardizes to lowercase snake_case."""
    col_str = str(col).strip()
    nfkd = unicodedata.normalize("NFKD", col_str)
    no_accent = "".join([c for c in nfkd if not unicodedata.combining(c)])
    return (
        no_accent.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .replace(".", "")
    )

def generate_complete_retail_base(folder: str = ZIP_FOLDER) -> None:
    """
    Processes multiple Central Bank ZIP files using an Out-of-Core architecture, 
    applying strict filters, numerical downcasting, and incremental Parquet exportation.
    """
    zip_files = sorted(glob.glob(os.path.join(folder, "*.zip")))

    if not zip_files:
        logger.error(f"No .zip files found in directory: '{folder}'")
        return None

    print("=" * 95)
    print(f"ETL SCR/BCB: FULL OUT-OF-CORE INGESTION OF {len(zip_files)} COMPRESSED FILES")
    print("=" * 95)

    writer: Optional[pq.ParquetWriter] = None
    total_files_processed = 0
    total_files_rejected = 0
    total_records_validated = 0

    for zip_file in zip_files:
        zip_name = os.path.basename(zip_file)
        print(f"\n>>> Processing compressed file: {zip_name}")

        try:
            with zipfile.ZipFile(zip_file, "r") as z:
                for csv_name in z.namelist():
                    if csv_name.lower().endswith(".csv"):
                        with z.open(csv_name) as f:
                            try:
                                df_chunk = pd.read_csv(f, sep=";", encoding="utf-8", low_memory=False)
                            except UnicodeDecodeError:
                                f.seek(0)
                                df_chunk = pd.read_csv(f, sep=";", encoding="latin1", low_memory=False)

                            df_chunk.columns = [normalize_column_name(c) for c in df_chunk.columns]

                            # 1. Strict Retail Filter (Pessoa Física)
                            col_cli = next((c for c in ["cliente", "tipo_cliente"] if c in df_chunk.columns), None)
                            if col_cli:
                                mask_pf = df_chunk[col_cli].astype(str).str.strip().str.upper().isin(["PF", "PESSOA FISICA", "PESSOA FÍSICA"])
                                df_chunk = df_chunk[mask_pf]
                            elif "cnae_ocupacao" in df_chunk.columns:
                                df_chunk = df_chunk[df_chunk["cnae_ocupacao"].notna()]
                            else:
                                logger.warning(f"Batch REJECTED: '{csv_name}'. No Retail (PF) identification column found.")
                                total_files_rejected += 1
                                continue

                            if len(df_chunk) == 0:
                                continue

                            # 2. Explicit Date Parsing
                            if "data_base" in df_chunk.columns:
                                df_chunk["data_base"] = pd.to_datetime(df_chunk["data_base"], format="%Y-%m-%d", errors="coerce")

                            # 3. Numerical Optimization (Downcast to float32)
                            found_monetary_cols = []
                            for col in df_chunk.columns:
                                if any(term in col for term in BALANCE_TERMS):
                                    if df_chunk[col].dtype == "object":
                                        clean_series = df_chunk[col].astype(str).str.strip().str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
                                        num_series = pd.to_numeric(clean_series, errors="coerce")
                                    else:
                                        num_series = pd.to_numeric(df_chunk[col], errors="coerce")
                                    
                                    if "carteira" in col:
                                        df_chunk[col] = num_series.astype(np.float32)
                                    else:
                                        df_chunk[col] = num_series.fillna(0.0).astype(np.float32)
                                    
                                    found_monetary_cols.append(col)

                            # 4. Survivorship Bias Correction (Total Exposure Mapping)
                            if found_monetary_cols:
                                df_chunk["carteira_total"] = df_chunk[found_monetary_cols].sum(axis=1, skipna=True)
                                # Strictly purges only if the total exposure is zero or null
                                df_chunk = df_chunk[df_chunk["carteira_total"].notna() & (df_chunk["carteira_total"] > 0)].copy()

                            # 5. Categorical Type Optimization
                            for cat_col in CATEGORICAL_COLUMNS:
                                if cat_col in df_chunk.columns:
                                    df_chunk[cat_col] = df_chunk[cat_col].astype("category")

                            # 6. Incremental Disk Writing (Prevents Out of Memory - OOM)
                            if len(df_chunk) > 0:
                                table = pa.Table.from_pandas(df_chunk)
                                
                                if writer is None:
                                    writer = pq.ParquetWriter(PARQUET_OUTPUT, table.schema, compression="snappy")
                                
                                writer.write_table(table)
                                
                                total_files_processed += 1
                                total_records_validated += len(df_chunk)
                                print(f"   -> {csv_name}: {len(df_chunk):,} records consolidated and saved.")

        except Exception as e:
            logger.error(f"Error processing '{zip_name}': {e}")
            continue

    if writer:
        writer.close()

    print("\n" + "=" * 95)
    print("CONSOLIDATED RETAIL CREDIT BASE (OUT-OF-CORE ARCHITECTURE)")
    print("=" * 95)
    print(f"Total validated and saved records: {total_records_validated:,}")
    print(f"Total processed files: {total_files_processed}")
    print(f"Total rejected files: {total_files_rejected}")
    print("=" * 95)

if __name__ == "__main__":
    generate_complete_retail_base()